**ANÁLISE DE DADOS - MINI PROJETO AVALIATIVO - BASE DE DADOS VAREJO**

**Aluno: Henrique da Silveira** 
**Data: 02-06-2026**

**IMPORTAÇÕES INICIAIS**

In [ ]:
#Importação de bibliotecas e exibição mais legível

import pandas as pd
import numpy as np
import sys
import warnings


#Importação das funções utils

sys.path.append('..')
from utils.funcoes_varejo import *


# Configurações para deixar a exibição mais legível
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)   # Mostrar todas as colunas
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 casas decimais

#Importação base de dados
df = pd.read_csv('../data/Base Varejo.csv', sep=';')

#Realiza disgnótico inicial dos dados
diagnostico(df,"Dados Varejo")

In [ ]:
#Realiza uma cópia do data frame original para realizar a limpeza e tratamento
df_tratamento = df.copy()

In [ ]:
#Corrige tipo de dados da coluna data

# Texto → data (dayfirst=True para datas no formato brasileiro DD/MM/YYYY)
df_tratamento['DATA'] = pd.to_datetime(df_tratamento['DATA'],format='%d/%m/%Y', dayfirst=True, errors='coerce')
print(df_tratamento.dtypes)

In [ ]:
# Limpar colunas vazia encontradas
df_tratamento = df_tratamento.dropna(axis=1, how='all')

#Exibe para confirmar que as colunas foram deletadas
print(df_tratamento.columns.tolist())

** TRANSFORMÇÕES / LIMPEZA DE DADOS**

In [ ]:
#Substitui valores nulos que não são dectataveis para detectaveis com pandas
df_tratamento= df_tratamento.replace({'NULL': np.nan, 'N/A': np.nan, '': np.nan, '#N/D': np.nan})

#Verifica se há espaços vazios que também não são nativamente detectáveis
for col in df_tratamento.columns:
    qtd = (df_tratamento[col].astype(str).str.strip() == '').sum()
    if qtd > 0:
        #Mostra que encontrou e a quantidade
        print(f'{col}: {qtd} VALORES VAZIOS ENCONTRADOS')
        
        #Realiza o tratamento destes valores caso sejam encontrados
        df_tratamento[col] = df_tratamento[col].replace(r'^\s*$', np.nan, regex=True)        
    else:        
        print(f'{col}: {qtd} Não há valores com espaços vazios')

In [ ]:
#Depois das substituições, realiza soma dos nulos detectáveis 
qtd_nulos = (df_tratamento.isnull().sum())

#Verifica se constam nulos após replace e se há dados que precisam de tratamento
for col in df_tratamento.columns:
     if qtd_nulos[col] > 0:
          print(f"{col}: {qtd_nulos[col]} VALORES NULOS ENCONTRADOS!")          
     else:
          print(f"{col}: SEM NULOS PARA TRATAR")

**TRATAMENTO DE VALORES NULOS / AUSENTES**

In [ ]:
#TRATAMENTO PRODUTOS SEM NOME
df_tratamento['PR_NOME'] = df_tratamento["PR_CAT"].fillna('NÃO INFORMADO')

#TRATAMENTO DE PRODUTOS SEM CATEGORIA
df_tratamento['PR_CAT'] = df_tratamento["PR_CAT"].fillna('SEM CATEGORIA')

AGRUPAMENTOS

In [ ]:
#PADRÃO 1: Categoria de Produto por Gênero
print("\n--- PADRÃO 1: Relação de Produto por Gênero ---")

# Agrupando por Gênero e Categoria do Produto
consumo_genero = df_tratamento.groupby(['PR_CAT', 'CL_GENERO']).size().reset_index(name='Total')
print(consumo_genero.to_string(index=False))

In [ ]:
#PADRÃO 2: Estrutura Familiar vs. Preferência de Categoria

print("\n--- PADRÃO 2: Estrutura Familiar vs. Categorias ---")

# Agrupamento combinando 3 variáveis: Estado Civil, Filhos e Categoria do Produto
padrao_familiar = df_tratamento.groupby(['CL_EC', 'CL_FHL', 'PR_CAT']).size().reset_index(name='Quantidade_Vendida')

# Combinações mais frequentes desse agrupamento (TOP 15)
print(padrao_familiar.sort_values(by='Quantidade_Vendida', ascending=False).head(15))

In [ ]:
#PADRÃO 3: Consumo por Segmento Econômico e Período
print("\n--- PADRÃO 3: Segmento Econômico vs. Tempo ---")

# Confirma o formato correto da data
df['DATA'] = pd.to_datetime(df['DATA'], dayfirst=True)

# Extraindo o dia da semana 
df['DIA'] = df['DATA'].dt.day_name(locale='pt_BR')

# Agrupa Classe Econômica com o Dia da Semana para ver o fluxo de compras
fluxo_segmento = df.groupby(['CL_SEG', 'DIA'])['CO_ID'].count().unstack(fill_value=0)

#Corrige visualização
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

print("\nVolume de compras por Segmento e Dia da Semana:")
print(fluxo_segmento)

**Análise de dados estatisticos - Números de Filhos dos cliente**

In [ ]:
# Contagem Geral de dados
print('TOTAL DE REGISTROS:')
print(df_tratamento['CL_FHL'].count())

# Analise de Média, Mediana e Moda relacionadas ao número de filhos dos clientes
print(f"Média de filhos: {df['CL_FHL'].mean():.2f}")   

print(f"Mediana: {df['CL_FHL'].median()}")   

# Retorna o valor mais comum de filhos
print(f"Moda: {df['CL_FHL'].mode()[0]}")  

#Quantidade máxima de filhos informada
print(f"Máximo: {df['CL_FHL'].max()}")

#Porcentagem de clientes com vs sem filhos
print(f"Clientes sem filhos: {(df['CL_FHL'] == 0).mean()*100:.2f}%")
print(f"Clientes com filhos: {(df['CL_FHL'] > 0).mean()*100:.2f}%")

**SALVAMENTO DO DATAFRAME LIMPO**

In [ ]:
# Salva o DataFrame como um arquivo CSV
df_tratamento.to_csv('../df_limpo.csv', index=False, encoding='utf-8-sig', sep=';')
print("Arquivo CSV salvo!")

Insgights Obtidos:

- 💡 #PADRÃO 1: Em todas as categorias de produtos, as mulheres são as principais compradoras, mostrando que são o públivo de maior interesse e que pode ser trabalhado formas de atrair mais compradores masculinos.

- 💡 #PADRÃO 2: Os clientes das primeiras posições da lista são estruturas familiares sem filhos, portanto, promoções focadas em "ITENS FAMILIARES" ou de grandes volumes podem não ser atrativas.

- 💡 #PADRÃO 3: Em todos os dias da semana, a classe B é o publico que mais compra, portanto deve ser o principal alvo do varejo.
 A classe A costuma ser mais frequente as quartas, verificar que tipo de oferta é oferecidas as quartas e aplica-las em outros dias da semana onde se deseja que esse público seja mais frequente.

- Na análise estatistica relizada sobre os filhos, percebe-se que as familias não são grandes.